## In-Class Lecture 37-1 - Thermal Fault Protection

In this lecture, you will write the four building blocks of a fault protection system: a voter for three redundant temperature sensors, a leaky bucket counter, a hysteresis heater controller, and an event report log. Twenty tests check each one. Then a mission driver connects all four and flies a thermal scenario with injected faults, and your event log tells the story of what happened.

### Reviewing Opening your Environment on the Notre Dame Machines

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your In-Class Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir inclass37
    cd inclass37

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/inclass/lec37/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    fault_protection.c  fault_protection.h  Makefile  test_fp.c  thermal.csv  thermal_run.c

### In-Class Reading Discussion

<font color = "red">Class Introduction Question #1 - In <code>tmr_vote.c</code>, why did the voter mask the fault in Case 2 but return the wrong value in Case 3? What does it do in Case 4, and why is that better than picking a value?</font>

<font color = "red">Class Introduction Question #2 - In Case 5, three separate sensors were used, and the voter still accepted a wrong value. What is hidden simplex, and how did it destroy Ariane 5?</font>

<font color = "red">Class Introduction Question #3 - Why did scrubbing once a week barely help, while scrubbing every hour prevented every failure in the simulation?</font>

<font color = "red">Class Introduction Question #4 - In <code>leaky_bucket.c</code>, why did the naive check trip at sample 2 while the bucket waited until sample 22? What happens to the response time if you raise <code>TRIP_LEVEL</code>?</font>

<font color = "red">Class Introduction Question #5 - Explain why the heater without hysteresis switched 503 times. What did hysteresis cost in exchange for reducing that to 53?</font>

<font color = "red">Class Introduction Question #6 - Why should a watchdog never be petted from a timer interrupt? Describe the four steps of the escalation ladder.</font>

### The Files

<ul>
    <li><code>fault_protection.h</code> - the structures and the four functions you will write. Read the comment above each function carefully: it is the specification.</li>
    <li><code>fault_protection.c</code> - your code. Each function compiles but returns a placeholder.</li>
    <li><code>test_fp.c</code> - 20 tests. You do not need to change it.</li>
    <li><code>thermal_run.c</code> and <code>thermal.csv</code> - the mission driver and a 29-second thermal scenario. Each line of the file is a time in milliseconds followed by the three channels, in hundredths of a degree.</li>
</ul>

Run the tests and the mission before you write anything:

    make test
    make mission

Five tests pass by coincidence. The mission prints an empty event log, because nothing is recorded yet.

### Step 1 - <code>vote</code> (Tests 1 to 6)

Write the majority voter from Reading 37-1, with one addition: it must report <b>which</b> channel was outvoted, so the ground team knows which sensor to suspect.
<ul>
    <li>All three agree: store the value, set <code>*dissenter</code> to <code>NO_DISSENT</code>, return <code>VOTE_OK</code>.</li>
    <li>Exactly two agree: store their value, set <code>*dissenter</code> to the index of the odd one out (0, 1, or 2), return <code>VOTE_OK</code>.</li>
    <li>No two agree, or any pointer is <code>NULL</code>: return <code>VOTE_NO_QUORUM</code>.</li>
</ul>

<b>Hint:</b> check the case where all three agree <i>first</i>. Why does the order of your checks matter?

### Step 2 - <code>bucket_update</code> (Tests 7 to 12)

Write the leaky bucket from Reading 37-2, using the fields of the structure instead of constants:
<ul>
    <li>A bad sample adds <code>fill</code>, but the level never goes above <code>max</code>.</li>
    <li>A good sample removes <code>leak</code>, but the level never goes below 0.</li>
    <li>Return 1 if the level is at or above <code>trip</code>, 0 if not, and -1 if <code>b</code> is <code>NULL</code>.</li>
</ul>

### Step 3 - <code>heater_command</code> (Tests 13 to 16)

Write the hysteresis rule from <code>thermostat.c</code>. Below <code>on_below</code>, the heater is on. Above <code>off_above</code>, it is off. In between, return the current state, unchanged. This one is short.

### Step 4 - <code>evr_add</code> (Tests 17 to 20)

Event reports go in a <b>ring buffer</b>: a fixed array where, once it is full, each new event replaces the oldest one. Flight software works this way because memory is fixed (Rule 3), and when space runs out, the newest events are usually the most important.
<ol>
    <li>Do nothing if <code>log</code> or <code>msg</code> is <code>NULL</code>.</li>
    <li>Write the time, severity, and message into <code>log->entries[log->next]</code>. Copy the message with <code>strncpy</code> and <b>always</b> set the last character to <code>'\0'</code>, as in the command queue from Lecture 34. Test 18 sends a message that is too long.</li>
    <li>Advance <code>next</code>, wrapping back to 0 at <code>EVR_CAPACITY</code>. The <code>%</code> operator does this in one line.</li>
    <li>Increase <code>count</code>, but never past <code>EVR_CAPACITY</code>. Always increase <code>total</code>.</li>
</ol>

### Step 5 - Fly the Mission

    make mission

Read your event log from top to bottom and match each event to the scenario:
<ul>
    <li>When does the heater turn on and off? Why does it not switch on every sample near 20 C?</li>
    <li>At 5000 ms and 12000 ms, one channel was outvoted. Look at the bad values. One is a bit flip, as in <code>tmr_vote.c</code>. What might the other one be?</li>
    <li>At 9000 ms, there was no quorum. Why didn't that single sample cause safe mode?</li>
    <li>At 20000 ms, the temperature reads 46.20 C, above the 45.00 C limit. Why didn't <i>that</i> cause safe mode, and what did?</li>
</ul>

<code>make mission</code> ends with <code>Error 1</code>, because the driver correctly returns <code>EXIT_FAILURE</code> when the spacecraft enters safe mode.

### Discussion

<ol>
    <li>Channel 2 reported -320.00 C at 12000 ms and was outvoted. If it reported that on <i>every</i> sample from then on, every vote would still succeed. Is the system still fault tolerant? What would you want the software to do?</li>
    <li>The ring buffer overwrites the oldest events when it is full. Can you think of an event you would never want overwritten? How might you protect it?</li>
    <li>The heater turned off at 20000 ms because of a single spike to 46.20 C. Should the heater controller also ride over transients? What could go wrong if it did?</li>
</ol>

## Compilation Run

    make test
    make mission
    make analyze

When you are finished, all 20 tests print <code>PASS</code>, the mission log shows every event from the scenario ending in safe mode, and <code>make analyze</code> reports no errors.